WILDLIFE SPECIES DISTRIBUTION ANALYSIS

  IMPORT LIBRARIES

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import io

warnings.filterwarnings('ignore')

# Set global plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10
})

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


DATA LOADING

In [ ]:
from google.colab import files
import io

# This will show a "Choose File" button in Colab — click it and upload your CSV
uploaded = files.upload()

# Read the uploaded file into a DataFrame
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"✅ File '{filename}' loaded successfully.")
print(f"   Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"\nFirst 5 rows:")
df_raw.head()
# Fix ThreatScore BEFORE any analysis

df['ThreatLevel'] = df['ThreatLevel'].astype(str).str.strip().str.title()

mapping = {
    'Low': 1,
    'Medium': 2,
    'High': 3,
    'Critical': 4
}

df['ThreatScore'] = df['ThreatLevel'].map(mapping)

df = df.dropna(subset=['ThreatScore'])

Saving wildlife_species_uncleaned_dataset.csv to wildlife_species_uncleaned_dataset (2).csv
✅ File 'wildlife_species_uncleaned_dataset (2).csv' loaded successfully.
   Shape: 7200 rows × 8 columns

First 5 rows:


DATA UNDERSTANDING

In [ ]:
print("\n" + "="*60)
print("2.1  DATASET SHAPE")
print("="*60)
print(f"Rows: {df_raw.shape[0]}   |   Columns: {df_raw.shape[1]}")

print("\n" + "="*60)
print("2.2  DATA TYPES & BASIC INFO")
print("="*60)
df_raw.info()

print("\n" + "="*60)
print("2.3  SUMMARY STATISTICS (numeric)")
print("="*60)
print(df_raw.describe())

print("\n" + "="*60)
print("2.4  SUMMARY STATISTICS (object columns)")
print("="*60)
print(df_raw.describe(include='object'))

print("\n" + "="*60)
print("2.5  MISSING VALUES")
print("="*60)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df  = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

print("\n" + "="*60)
print("2.6  DUPLICATE ROWS")
print("="*60)
print(f"Total duplicates: {df_raw.duplicated().sum()}")

print("\n" + "="*60)
print("2.7  UNIQUE VALUES PER COLUMN")
print("="*60)
for col in df_raw.columns:
    print(f"  {col:22s}: {df_raw[col].nunique()} unique values")


2.1  DATASET SHAPE
Rows: 7200   |   Columns: 8

2.2  DATA TYPES & BASIC INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7200 entries, 0 to 7199
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Species             6611 non-null   object 
 1   Region              6156 non-null   object 
 2   HabitatType         6271 non-null   object 
 3   Year                6979 non-null   float64
 4   PopulationEstimate  7200 non-null   int64  
 5   ThreatLevel         6188 non-null   object 
 6   Latitude            7200 non-null   float64
 7   Longitude           7200 non-null   float64
dtypes: float64(3), int64(1), object(4)
memory usage: 450.1+ KB

2.3  SUMMARY STATISTICS (numeric)
              Year  PopulationEstimate     Latitude    Longitude
count  6979.000000         7200.000000  7200.000000  7200.000000
mean   2009.948130        50165.178750    -1.338152    -0.421049
std       8.904989        29169.

 DATA CLEANING

In [ ]:
df = df_raw.copy()
print("\n🔧 Starting data cleaning …")

# ── 3.1  Remove duplicates ───────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f"  [3.1] Duplicates removed  : {before - len(df)}")

# ── 3.2  Fix text inconsistencies (case & whitespace) ────────
text_cols = ['Species', 'Region', 'HabitatType', 'ThreatLevel']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    # Replace 'Nan' strings produced by .astype(str) → actual NaN
    df[col] = df[col].replace('Nan', np.nan)
print("  [3.2] Text case & whitespace fixed.")

# ── 3.3  Handle invalid numeric values ───────────────────────
# Negative population → set to NaN
neg_pop = (df['PopulationEstimate'] < 0).sum()
df.loc[df['PopulationEstimate'] < 0, 'PopulationEstimate'] = np.nan
print(f"  [3.3] Negative PopulationEstimate set to NaN : {neg_pop}")

# Invalid Latitude (outside -90 to 90) → set to NaN
inv_lat = (~df['Latitude'].between(-90, 90, inclusive='both')).sum()
df.loc[~df['Latitude'].between(-90, 90, inclusive='both'), 'Latitude'] = np.nan
print(f"  [3.3] Invalid Latitude set to NaN            : {inv_lat}")

# Invalid Longitude (outside -180 to 180) → set to NaN
inv_lon = (~df['Longitude'].between(-180, 180, inclusive='both')).sum()
df.loc[~df['Longitude'].between(-180, 180, inclusive='both'), 'Longitude'] = np.nan
print(f"  [3.3] Invalid Longitude set to NaN           : {inv_lon}")

# ── 3.4  Handle missing values ───────────────────────────────
# PopulationEstimate → fill with median per species
df['PopulationEstimate'] = df.groupby('Species')['PopulationEstimate'] \
                             .transform(lambda x: x.fillna(x.median()))
# If still NaN (species had all-NaN), fill with overall median
df['PopulationEstimate'].fillna(df['PopulationEstimate'].median(), inplace=True)

# ThreatLevel → fill with mode per species
df['ThreatLevel'] = df.groupby('Species')['ThreatLevel'] \
                      .transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown'))
df['ThreatLevel'].fillna('Unknown', inplace=True)

# Latitude & Longitude → fill with mean per Region
for coord in ['Latitude', 'Longitude']:
    df[coord] = df.groupby('Region')[coord] \
                  .transform(lambda x: x.fillna(x.mean()))
    df[coord].fillna(df[coord].mean(), inplace=True)

# Year → fill with median
df['Year'].fillna(df['Year'].median(), inplace=True)

print("  [3.4] Missing values imputed.")

# ── 3.5  Data type conversion ────────────────────────────────
df['Year']              = df['Year'].astype(int)
df['PopulationEstimate'] = df['PopulationEstimate'].astype(float).round(0).astype(int)
df['Latitude']          = df['Latitude'].astype(float).round(4)
df['Longitude']         = df['Longitude'].astype(float).round(4)
print("  [3.5] Data types converted.")

# ── 3.6  Final verification ──────────────────────────────────
print(f"\n✅ Cleaning complete  →  {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Remaining nulls : {df.isnull().sum().sum()}")
print(df.dtypes)


🔧 Starting data cleaning …
  [3.1] Duplicates removed  : 200
  [3.2] Text case & whitespace fixed.
  [3.3] Negative PopulationEstimate set to NaN : 0
  [3.3] Invalid Latitude set to NaN            : 673
  [3.3] Invalid Longitude set to NaN           : 703
  [3.4] Missing values imputed.
  [3.5] Data types converted.

✅ Cleaning complete  →  7000 rows × 8 columns
   Remaining nulls : 2484
Species                object
Region                 object
HabitatType            object
Year                    int64
PopulationEstimate      int64
ThreatLevel            object
Latitude              float64
Longitude             float64
dtype: object


FEATURE ENGINEERING

In [ ]:
print("\n⚙️  Feature engineering …")

# ── 4.1  ThreatScore : ordinal mapping ───────────────────────
threat_map = {'Low': 1, 'Moderate': 2, 'High': 3, 'Critical': 4, 'Extinct': 5, 'Unknown': 0}
df['ThreatScore'] = df['ThreatLevel'].map(threat_map).fillna(0).astype(int)
print("  [4.1] ThreatScore created.")

# ── 4.2  PopulationTrend : % change vs previous year per species ─
df_sorted = df.sort_values(['Species', 'Year'])
df_sorted['PopulationTrend'] = df_sorted.groupby('Species')['PopulationEstimate'] \
                                         .pct_change() * 100
df_sorted['PopulationTrend'] = df_sorted['PopulationTrend'].round(2)
df = df_sorted.copy()
print("  [4.2] PopulationTrend (% YoY change) created.")

# ── 4.3  ConservationUrgency : composite score ───────────────
df['PopLog'] = np.log1p(df['PopulationEstimate'])
pop_log_max  = df['PopLog'].max()
# Higher urgency = higher threat + lower (log) population
df['ConservationUrgency'] = (
    (df['ThreatScore'] / 5) * 0.6 +
    (1 - df['PopLog'] / pop_log_max) * 0.4
).round(4)
df.drop(columns=['PopLog'], inplace=True)
print("  [4.3] ConservationUrgency (composite 0–1) created.")

# ── 4.4  DecadeGroup ─────────────────────────────────────────
df['DecadeGroup'] = df['Year'].apply(lambda y: f"{(y // 5) * 5}s")
print("  [4.4] DecadeGroup (5-yr bucket) created.")

# ── 4.5  IsEndangered flag ───────────────────────────────────
df['IsEndangered'] = df['ThreatScore'].apply(lambda s: 1 if s >= 3 else 0)
print("  [4.5] IsEndangered flag (ThreatScore ≥ 3) created.")

print("\n✅ Feature engineering complete.")
print(df[['Species','ThreatScore','PopulationTrend','ConservationUrgency',
          'DecadeGroup','IsEndangered']].head(10).to_string())



⚙️  Feature engineering …
  [4.1] ThreatScore created.
  [4.2] PopulationTrend (% YoY change) created.
  [4.3] ConservationUrgency (composite 0–1) created.
  [4.4] DecadeGroup (5-yr bucket) created.
  [4.5] IsEndangered flag (ThreatScore ≥ 3) created.

✅ Feature engineering complete.
       Species  ThreatScore  PopulationTrend  ConservationUrgency DecadeGroup  IsEndangered
227   Elephant            0              NaN               0.0020       1995s             0
339   Elephant            0            -6.26               0.0042       1995s             0
476   Elephant            0           -46.46               0.0259       1995s             0
569   Elephant            0           -48.70               0.0491       1995s             0
593   Elephant            0           -58.59               0.0797       1995s             0
1146  Elephant            0           649.59               0.0098       1995s             0
1259  Elephant            0           -25.13               0.0198     

EXPLORATORY DATA ANALYSIS (EDA)

1 ─Species count bar chart

In [ ]:
TEMPLATE = 'plotly_white'

species_counts = df['Species'].value_counts().reset_index()
species_counts.columns = ['Species', 'Count']

fig = px.bar(species_counts, x='Species', y='Count',
             color='Species', text='Count',
             title='VIZ 1 · Record Count per Species',
             template=TEMPLATE,
             hover_data={'Species': True, 'Count': True})
fig.update_traces(textposition='outside', hovertemplate='<b>%{x}</b><br>Records: %{y}<extra></extra>')
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

2 -ThreatLevel distribution (pie chart)

In [ ]:
threat_counts = df['ThreatLevel'].value_counts().reset_index()
threat_counts.columns = ['ThreatLevel', 'Count']

fig = px.pie(threat_counts, names='ThreatLevel', values='Count',
             title='VIZ 2 · Threat Level Distribution',
             color='ThreatLevel',
             color_discrete_map={
                 'Low': '#2ca02c', 'Moderate': '#ffbf00',
                 'High': '#ff7f0e', 'Critical': '#d62728',
                 'Extinct': '#7f0000', 'Unknown': '#aec7e8'
             },
             template=TEMPLATE,
             hole=0)
fig.update_traces(
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Share: %{percent}<extra></extra>'
)
fig.show()

3 ─ Population by Region (Box)

In [ ]:
region_order = df.groupby('Region')['PopulationEstimate'].median().sort_values(ascending=False).index.tolist()

fig = px.box(df, x='Region', y='PopulationEstimate',
             color='Region', category_orders={'Region': region_order},
             title='VIZ 3 · Population Estimate Distribution by Region',
             log_y=True, template=TEMPLATE,
             hover_data=['Species', 'Year', 'ThreatLevel'])
fig.update_traces(hovertemplate='<b>%{x}</b><br>Population: %{y:,}<br>Species: %{customdata[0]}<br>Year: %{customdata[1]}<br>Threat: %{customdata[2]}<extra></extra>')
fig.update_layout(showlegend=False)
fig.show()


 VIZ 4 ─ Population Over Years (Line)

In [ ]:
pop_trend = df.groupby('Year').agg(
    AvgPopulation=('PopulationEstimate', 'mean'),
    TotalObservations=('Species', 'count')
).reset_index()

fig = px.line(pop_trend, x='Year', y='AvgPopulation',
              markers=True, title='VIZ 4 · Average Population Estimate Over Years',
              template=TEMPLATE,
              hover_data={'Year': True, 'AvgPopulation': ':.0f', 'TotalObservations': True})
fig.update_traces(
    line=dict(color='steelblue', width=2.5),
    marker=dict(size=8),
    hovertemplate='<b>Year: %{x}</b><br>Avg Population: %{y:,.0f}<br>Observations: %{customdata[1]}<extra></extra>'
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()


 5 ─ Habitat Distribution (Donut)

In [ ]:
hab_counts = df['HabitatType'].value_counts().reset_index()
hab_counts.columns = ['HabitatType', 'Count']

fig = px.pie(hab_counts, names='HabitatType', values='Count',
             title='VIZ 5 · Habitat Type Distribution (Donut)',
             template=TEMPLATE, hole=0.45)
fig.update_traces(
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Share: %{percent}<extra></extra>'
)
fig.show()

6─ Average Population per Species (Horizontal Bar)

In [ ]:
avg_pop = df.groupby('Species')['PopulationEstimate'] \
             .mean().reset_index().sort_values('PopulationEstimate', ascending=True)
avg_pop.columns = ['Species', 'AvgPopulation']

fig = px.bar(avg_pop, x='AvgPopulation', y='Species',
             orientation='h', color='AvgPopulation',
             color_continuous_scale='Blues',
             text='AvgPopulation',
             title='VIZ 6· Average Population Estimate per Species',
             template=TEMPLATE)
fig.update_traces(
    texttemplate='%{text:,.0f}', textposition='outside',
    hovertemplate='<b>%{y}</b><br>Avg Population: %{x:,.0f}<extra></extra>'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()
print("Insight → Species with critically low average populations need urgent attention.")


Insight → Species with critically low average populations need urgent attention.


7─ Heatmap: Species × Habitat Average Population

In [ ]:
pivot_pop = df.pivot_table(values='PopulationEstimate',
                            index='Species',
                            columns='HabitatType',
                            aggfunc='mean').round(0)

fig = go.Figure(data=go.Heatmap(
    z=pivot_pop.values,
    x=pivot_pop.columns.tolist(),
    y=pivot_pop.index.tolist(),
    colorscale='Blues',
    text=pivot_pop.values,
    texttemplate='%{text:,.0f}',
    hovertemplate='Species: <b>%{y}</b><br>'
                  'Habitat: <b>%{x}</b><br>'
                  'Avg Population: %{z:,.0f}<extra></extra>',
    colorbar=dict(title='Avg Population')
))
fig.update_layout(
    title='VIZ 7 · Average Population — Species × Habitat',
    template=TEMPLATE, height=500
)
fig.show()
print("Insight → Which species-habitat combinations have the most/least viable populations.")


Insight → Which species-habitat combinations have the most/least viable populations.


8─ ThreatLevel by Region (Stacked Bar)

In [ ]:
threat_region = df.groupby(['Region', 'ThreatLevel']).size().reset_index(name='Count')
threat_order  = ['Low', 'Moderate', 'High', 'Critical', 'Extinct', 'Unknown']

fig = px.bar(threat_region, x='Region', y='Count', color='ThreatLevel',
             barmode='stack',
             title='VIZ 8 · Threat Level Breakdown by Region',
             template=TEMPLATE,
             category_orders={'ThreatLevel': threat_order},
             color_discrete_map={
                 'Low': '#2ca02c', 'Moderate': '#ffbf00',
                 'High': '#ff7f0e', 'Critical': '#d62728',
                 'Extinct': '#7f0000', 'Unknown': '#aec7e8'
             })
fig.update_traces(
    hovertemplate='<b>%{x}</b><br>Threat: %{fullData.name}<br>Count: %{y}<extra></extra>'
)
fig.update_layout(xaxis_tickangle=-25)
fig.show()
print("Insight → Regions with the highest concentration of Critical/Extinct observations.")


Insight → Regions with the highest concentration of Critical/Extinct observations.


9 ── Population per Species Over Years (Multi-line)

In [ ]:
pop_species_yr = df.groupby(['Year', 'Species'])['PopulationEstimate'] \
                    .mean().reset_index()

fig = px.line(pop_species_yr, x='Year', y='PopulationEstimate',
              color='Species', markers=True,
              title='VIZ 9 · Population Trend by Species Over Years',
              template=TEMPLATE)
fig.update_traces(
    marker=dict(size=6),
    hovertemplate='<b>%{fullData.name}</b><br>Year: %{x}<br>Avg Population: %{y:,.0f}<extra></extra>'
)
fig.update_layout(xaxis=dict(dtick=1), legend=dict(title='Species'))
fig.show()
print("Insight → Species showing consistent decline vs those that are recovering.")

Insight → Species showing consistent decline vs those that are recovering.


10 ─ Conservation Urgency Distribution (Histogram)

In [ ]:
fig = px.histogram(df, x='ConservationUrgency', nbins=35,
                   color_discrete_sequence=['coral'],
                   title='VIZ 10 · Distribution of Conservation Urgency Score',
                   template=TEMPLATE, marginal='violin')
fig.update_traces(
    hovertemplate='Urgency: %{x:.3f}<br>Count: %{y}<extra></extra>'
)
fig.update_layout(
    xaxis_title='Conservation Urgency (0 = Least → 1 = Most Urgent)',
    yaxis_title='Frequency'
)
fig.show()
print("Insight → Bimodal urgency split highlights two distinct conservation priority groups.")

Insight → Bimodal urgency split highlights two distinct conservation priority groups.


11 ─ Population by ThreatLevel (Box)

In [ ]:
threat_ord = ['Low', 'Moderate', 'High', 'Critical', 'Extinct', 'Unknown']
threat_ord = [t for t in threat_ord if t in df['ThreatLevel'].unique()]

fig = px.box(df, x='ThreatLevel', y='PopulationEstimate',
             color='ThreatLevel', log_y=True,
             category_orders={'ThreatLevel': threat_ord},
             title='VIZ 11 · Population Estimate by Threat Level',
             template=TEMPLATE,
             hover_data=['Species', 'Region', 'Year'],
             color_discrete_map={
                 'Low': '#2ca02c', 'Moderate': '#ffbf00',
                 'High': '#ff7f0e', 'Critical': '#d62728',
                 'Extinct': '#7f0000', 'Unknown': '#aec7e8'
             })
fig.update_traces(
    hovertemplate='<b>%{x}</b><br>Population: %{y:,}<br>'
                  'Species: %{customdata[0]}<br>'
                  'Region: %{customdata[1]}<br>'
                  'Year: %{customdata[2]}<extra></extra>'
)
fig.update_layout(showlegend=False)
fig.show()
print("Insight → Clear inverse relationship between threat level and population size.")

Insight → Clear inverse relationship between threat level and population size.


12─ Conservation Urgency by Habitat (Bar)

In [ ]:
hab_urgency = df.groupby('HabitatType').agg(
    AvgUrgency=('ConservationUrgency', 'mean'),
    Count=('Species', 'count'),
    AvgPopulation=('PopulationEstimate', 'mean')
).reset_index().sort_values('AvgUrgency', ascending=False).round(3)

fig = px.bar(hab_urgency, x='HabitatType', y='AvgUrgency',
             color='AvgUrgency', text='AvgUrgency',
             color_continuous_scale='Reds',
             title='VIZ 12 · Average Conservation Urgency by Habitat Type',
             template=TEMPLATE,
             hover_data={
                 'HabitatType': True,
                 'AvgUrgency': ':.3f',
                 'Count': True,
                 'AvgPopulation': ':.0f'
             })
fig.update_traces(
    texttemplate='%{text:.3f}', textposition='outside',
    hovertemplate='<b>%{x}</b><br>'
                  'Avg Urgency: %{y:.3f}<br>'
                  'Records: %{customdata[1]}<br>'
                  'Avg Population: %{customdata[2]:,.0f}<extra></extra>'
)
fig.update_layout(coloraxis_showscale=False, yaxis_range=[0, 1])
fig.show()
print("Insight → Habitats ranked by conservation urgency to prioritize protection efforts.")

Insight → Habitats ranked by conservation urgency to prioritize protection efforts.


13─ ThreatLevel per Habitat (Grouped Bar)

In [ ]:
threat_hab = df.groupby(['HabitatType', 'ThreatLevel']).size().reset_index(name='Count')
threat_order = ['Low', 'Moderate', 'High', 'Critical', 'Extinct', 'Unknown']

fig = px.bar(threat_hab, x='HabitatType', y='Count', color='ThreatLevel',
             barmode='group',
             title='VIZ 13 · Threat Level Distribution Across Habitat Types',
             template=TEMPLATE,
             category_orders={'ThreatLevel': threat_order},
             color_discrete_map={
                 'Low': '#2ca02c', 'Moderate': '#ffbf00',
                 'High': '#ff7f0e', 'Critical': '#d62728',
                 'Extinct': '#7f0000', 'Unknown': '#aec7e8'
             })
fig.update_traces(
    hovertemplate='<b>%{x}</b><br>Threat: %{fullData.name}<br>Count: %{y}<extra></extra>'
)
fig.update_layout(xaxis_tickangle=-25, legend_title='Threat Level')
fig.show()
print("Insight → Habitat types most associated with Critical and Extinct threat levels.")

Insight → Habitat types most associated with Critical and Extinct threat levels.


 14-Interactive Geo Map

In [ ]:
df_map = df.dropna(subset=['Latitude', 'Longitude']).copy()

fig = px.scatter_geo(
    df_map, lat='Latitude', lon='Longitude',
    color='ThreatLevel', symbol='ThreatLevel',
    hover_name='Species',
    hover_data={
        'Region': True,
        'HabitatType': True,
        'PopulationEstimate': True,
        'ThreatLevel': True,
        'Year': True,
        'ConservationUrgency': ':.3f',
        'Latitude': False,
        'Longitude': False
    },
    color_discrete_map={
        'Low': '#2ca02c', 'Moderate': '#ffbf00',
        'High': '#ff7f0e', 'Critical': '#d62728',
        'Extinct': '#7f0000', 'Unknown': '#aec7e8'
    },
    size='ConservationUrgency', size_max=14,
    projection='natural earth',
    title='VIZ 14 · Global Wildlife Species Distribution & Threat Levels',
    template=TEMPLATE
)
fig.update_traces(
    hovertemplate='<b>%{hovertext}</b><br>'
                  'Region: %{customdata[0]}<br>'
                  'Habitat: %{customdata[1]}<br>'
                  'Population: %{customdata[2]:,}<br>'
                  'Threat: %{customdata[3]}<br>'
                  'Year: %{customdata[4]}<br>'
                  'Urgency: %{customdata[5]:.3f}<extra></extra>'
)
fig.update_layout(
    geo=dict(
        showframe=False, showcoastlines=True, coastlinecolor='gray',
        showland=True, landcolor='lightyellow',
        showocean=True, oceancolor='lightcyan'
    ),
    legend_title_text='Threat Level',
    height=580
)
fig.show()
print("Insight → Geographic hotspots of Critical/Extinct species for targeted field intervention.")

Insight → Geographic hotspots of Critical/Extinct species for targeted field intervention.


DASHBOARD

In [2]:
# ══════════════════════════════════════════════════════════════════
# 🐾 INTERACTIVE WILDLIFE CONSERVATION DASHBOARD  (Google Colab)
# ══════════════════════════════════════════════════════════════════
# Enable Colab widget + plotly support
from google.colab import output as colab_output
colab_output.enable_custom_widget_manager()

import plotly.io as pio
pio.renderers.default = "colab"

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import numpy as np # Added for data preparation
import io # Added for data loading
from google.colab import files # Added for data loading

# ============================================================
# DATA LOADING (Making the dashboard self-contained)
# ============================================================
# This will show a "Choose File" button in Colab — click it and upload your CSV
# Ensure you upload the 'wildlife_species_uncleaned_dataset.csv' file.
uploaded = files.upload()

# Read the uploaded file into a DataFrame
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))

# ============================================================
# DATA CLEANING & FEATURE ENGINEERING (Making the dashboard self-contained)
# ============================================================
df = df_raw.copy()

# Data Cleaning from M4CTCzez_k5z
df.drop_duplicates(inplace=True)

text_cols = ['Species', 'Region', 'HabitatType', 'ThreatLevel']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace('Nan', np.nan)

df.loc[df['PopulationEstimate'] < 0, 'PopulationEstimate'] = np.nan
df.loc[~df['Latitude'].between(-90, 90, inclusive='both'), 'Latitude'] = np.nan
df.loc[~df['Longitude'].between(-180, 180, inclusive='both'), 'Longitude'] = np.nan

df['PopulationEstimate'] = df.groupby('Species')['PopulationEstimate'] \
                             .transform(lambda x: x.fillna(x.median()))
df['PopulationEstimate'].fillna(df['PopulationEstimate'].median(), inplace=True)
df['ThreatLevel'] = df.groupby('Species')['ThreatLevel'] \
                      .transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown'))
df['ThreatLevel'].fillna('Unknown', inplace=True)
for coord in ['Latitude', 'Longitude']:
    df[coord] = df.groupby('Region')[coord] \
                  .transform(lambda x: x.fillna(x.mean()))
    df[coord].fillna(df[coord].mean(), inplace=True)
df['Year'].fillna(df['Year'].median(), inplace=True)

df['Year']              = df['Year'].astype(int)
df['PopulationEstimate'] = df['PopulationEstimate'].astype(float).round(0).astype(int)
df['Latitude']          = df['Latitude'].astype(float).round(4)
df['Longitude']         = df['Longitude'].astype(float).round(4)

# Feature Engineering from ol6F16lF__Ed
threat_map = {'Low': 1, 'Moderate': 2, 'High': 3, 'Critical': 4, 'Extinct': 5, 'Unknown': 0}
df['ThreatScore'] = df['ThreatLevel'].map(threat_map).fillna(0).astype(int)

df_sorted = df.sort_values(['Species', 'Year'])
df_sorted['PopulationTrend'] = df_sorted.groupby('Species')['PopulationEstimate'] \
                                         .pct_change() * 100
df_sorted['PopulationTrend'] = df_sorted['PopulationTrend'].round(2)
df = df_sorted.copy()

df['PopLog'] = np.log1p(df['PopulationEstimate'])
pop_log_max  = df['PopLog'].max()
df['ConservationUrgency'] = (
    (df['ThreatScore'] / 5) * 0.6 +
    (1 - df['PopLog'] / pop_log_max) * 0.4
).round(4)
df.drop(columns=['PopLog'], inplace=True)

df['DecadeGroup'] = df['Year'].apply(lambda y: f"{(y // 5) * 5}s")

df['IsEndangered'] = df['ThreatScore'].apply(lambda s: 1 if s >= 3 else 0)


# ---------- Header ----------
display(HTML("""
<div style="background:linear-gradient(90deg,#0f4c3a,#2e8b57);
            padding:20px;border-radius:12px;color:white;text-align:center;
            margin-bottom:15px;font-family:Arial;">
  <h1 style="margin:0;">🐾 Wildlife Species Distribution Dashboard</h1>
  <p style="margin:5px 0 0;opacity:0.9;">
     Interactive analysis of species, threats, habitats & population trends
  </p>
</div>"""))

# ---------- Color map ----------
THREAT_COLORS = {
    'Low': '#2ca02c', 'Moderate': '#ffbf00', 'High': '#ff7f0e',
    'Critical': '#d62728', 'Extinct': '#7f0000', 'Unknown': '#aec7e8'
}

# ---------- Filter widgets ----------
region_filter = widgets.SelectMultiple(
    options=['All'] + sorted(df['Region'].dropna().unique().tolist()),
    value=('All',), description='Region:',
    layout=widgets.Layout(width='230px', height='120px')
)
habitat_filter = widgets.SelectMultiple(
    options=['All'] + sorted(df['HabitatType'].dropna().unique().tolist()),
    value=('All',), description='Habitat:',
    layout=widgets.Layout(width='230px', height='120px')
)
threat_filter = widgets.SelectMultiple(
    options=['All'] + sorted(df['ThreatLevel'].dropna().unique().tolist()),
    value=('All',), description='Threat:',
    layout=widgets.Layout(width='230px', height='120px')
)
year_min, year_max = int(df['Year'].min()), int(df['Year'].max())
year_filter = widgets.IntRangeSlider(
    value=[year_min, year_max], min=year_min, max=year_max, step=1,
    description='Year:', continuous_update=False,
    layout=widgets.Layout(width='500px')
)
run_btn = widgets.Button(description='🔄 Update Dashboard',
                         button_style='success',
                         layout=widgets.Layout(width='220px', height='40px'))
output_area = widgets.Output()

# ---------- KPI helper ----------
def kpi_card(label, value, color):
    return f"""
    <div style="background:{color};padding:18px;border-radius:10px;
                color:white;text-align:center;flex:1;margin:5px;
                box-shadow:0 2px 6px rgba(0,0,0,0.15);min-width:140px;">
      <div style="font-size:13px;opacity:0.9;">{label}</div>
      <div style="font-size:22px;font-weight:bold;margin-top:4px;">{value}</div>
    </div>"""

# ---------- Dashboard builder ----------
def build_dashboard(_=None):
    with output_area:
        clear_output(wait=True)

        # Apply filters
        dff = df.copy()
        if 'All' not in region_filter.value:
            dff = dff[dff['Region'].isin(region_filter.value)]
        if 'All' not in habitat_filter.value:
            dff = dff[dff['HabitatType'].isin(habitat_filter.value)]
        if 'All' not in threat_filter.value:
            dff = dff[dff['ThreatLevel'].isin(threat_filter.value)]
        dff = dff[(dff['Year'] >= year_filter.value[0]) &
                  (dff['Year'] <= year_filter.value[1])]

        if dff.empty:
            display(HTML("<h3 style='color:red;'>⚠️ No data matches the selected filters.</h3>"))
            return

        # KPIs
        total_records  = len(dff)
        unique_species = dff['Species'].nunique()
        avg_pop        = dff['PopulationEstimate'].mean()
        endangered_pct = (dff['ThreatLevel'].isin(['High','Critical','Extinct']).sum()
                          / len(dff) * 100)
        avg_urgency    = dff['ConservationUrgency'].mean()

        display(HTML(f"""
        <div style="display:flex;flex-wrap:wrap;margin-bottom:15px;">
          {kpi_card('Total Records', f'{total_records:,}', '#1f77b4')}
          {kpi_card('Unique Species', f'{unique_species}', '#2ca02c')}
          {kpi_card('Avg Population', f'{avg_pop:,.0f}', '#17becf')}
          {kpi_card('Endangered %', f'{endangered_pct:.1f}%', '#d62728')}
          {kpi_card('Avg Urgency', f'{avg_urgency:.3f}', '#ff7f0e')}
        </div>"""))

        # Row 1 : species count + threat pie
        sp_counts = dff['Species'].value_counts().reset_index()
        sp_counts.columns = ['Species', 'Count']
        th_counts = dff['ThreatLevel'].value_counts().reset_index()
        th_counts.columns = ['ThreatLevel', 'Count']

        fig1 = make_subplots(
            rows=1, cols=2,
            specs=[[{'type':'bar'}, {'type':'domain'}]],
            subplot_titles=('Records per Species', 'Threat Level Distribution')
        )
        fig1.add_trace(
            go.Bar(x=sp_counts['Species'], y=sp_counts['Count'],
                   marker_color='steelblue', text=sp_counts['Count'],
                   textposition='outside', name='Records'),
            row=1, col=1
        )
        fig1.add_trace(
            go.Pie(labels=th_counts['ThreatLevel'], values=th_counts['Count'],
                   hole=0.4,
                   marker=dict(colors=[THREAT_COLORS.get(t,'#999') for t in th_counts['ThreatLevel']])),
            row=1, col=2
        )
        fig1.update_layout(height=430, template='plotly_white',
                           title_text='📌 Species & Threat Overview')
        fig1.show()

        # Row 2 : population over years
        pop_trend = dff.groupby('Year')['PopulationEstimate'].mean().reset_index()
        fig2 = px.line(pop_trend, x='Year', y='PopulationEstimate', markers=True,
                       title='📈 Average Population Estimate Over Years',
                       template='plotly_white')
        fig2.update_traces(line=dict(color='seagreen', width=3),
                           marker=dict(size=9))
        fig2.update_layout(height=380)
        fig2.show()

        # Row 3 : habitat donut + region box
        hab_counts = dff['HabitatType'].value_counts().reset_index()
        hab_counts.columns = ['HabitatType', 'Count']
        fig3 = make_subplots(
            rows=1, cols=2,
            specs=[[{'type':'domain'}, {'type':'box'}]],
            subplot_titles=('Habitat Distribution', 'Population by Region (log)')
        )
        fig3.add_trace(
            go.Pie(labels=hab_counts['HabitatType'], values=hab_counts['Count'], hole=0.45),
            row=1, col=1
        )
        for region in dff['Region'].unique():
            fig3.add_trace(
                go.Box(y=dff[dff['Region']==region]['PopulationEstimate'],
                       name=str(region), boxmean=True),
                row=1, col=2
            )
        fig3.update_yaxes(type='log', row=1, col=2)
        fig3.update_layout(height=450, template='plotly_white',
                           title_text='🌿 Habitat & Regional View',
                           showlegend=False)
        fig3.show()

        # Row 4 : threat by region
        tr = dff.groupby(['Region','ThreatLevel']).size().reset_index(name='Count')
        fig4 = px.bar(tr, x='Region', y='Count', color='ThreatLevel',
                      barmode='stack', color_discrete_map=THREAT_COLORS,
                      title='🚨 Threat Level Breakdown by Region',
                      template='plotly_white')
        fig4.update_layout(height=420)
        fig4.show()

        # Row 5 : geo map
        df_map = dff.dropna(subset=['Latitude','Longitude'])
        if not df_map.empty:
            fig5 = px.scatter_geo(
                df_map, lat='Latitude', lon='Longitude',
                color='ThreatLevel', hover_name='Species',
                hover_data=['Region','HabitatType','PopulationEstimate','Year'],
                color_discrete_map=THREAT_COLORS,
                title='🗺️ Global Wildlife Observations Map',
                template='plotly_white'
            )
            fig5.update_layout(height=500,
                               geo=dict(showland=True, landcolor='rgb(243,243,243)',
                                        showcountries=True))
            fig5.show()

        # Row 6 : top endangered table
        top_end = (dff[dff['ThreatLevel'].isin(['High','Critical','Extinct'])]
                   .groupby('Species')
                   .agg(Records=('Species','count'),
                        AvgPop=('PopulationEstimate','mean'),
                        AvgUrgency=('ConservationUrgency','mean'))
                   .round(2)
                   .sort_values('AvgUrgency', ascending=False)
                   .head(10)
                   .reset_index())

        if not top_end.empty:
            display(HTML("<h3 style='color:#0f4c3a;'>🔥 Top 10 Most At-Risk Species (Filtered)</h3>"))
            display(top_end.style.background_gradient(cmap='Reds', subset=['AvgUrgency'])
                                 .format({'AvgPop':'{:,.0f}', 'AvgUrgency':'{:.3f}'}))

        print(f"\n✅ Dashboard updated · {total_records:,} records · "
              f"{unique_species} species · Years {year_filter.value[0]}–{year_filter.value[1]}")

# ---------- Bind controls ----------
run_btn.on_click(build_dashboard)

filter_box = widgets.HBox([region_filter, habitat_filter, threat_filter])
controls   = widgets.VBox([filter_box, year_filter, run_btn])
display(controls, output_area)

# Initial render
build_dashboard()

Saving wildlife_species_dataset.csv to wildlife_species_dataset.csv


/tmp/ipykernel_11745/91697160.py:51: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.



/tmp/ipykernel_11745/91697160.py:54: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or

Output()

 KEY INSIGHTS SUMMARY

In [ ]:
print("\n" + "="*65)
print("SECTION 6 — KEY INSIGHTS FROM WILDLIFE DISTRIBUTION ANALYSIS")
print("="*65)

# ── 6.1  Most Endangered Species ─────────────────────────────
most_endangered = df.groupby('Species').agg(
    CriticalCount   = ('ThreatLevel', lambda x: (x == 'Critical').sum()),
    ExtinctCount    = ('ThreatLevel', lambda x: (x == 'Extinct').sum()),
    AvgPopulation   = ('PopulationEstimate', 'mean'),
    AvgUrgency      = ('ConservationUrgency', 'mean'),
    EndangeredTotal = ('IsEndangered', 'sum'),
    TotalRecords    = ('Species', 'count')
).round(2)

most_endangered['EndangeredRate%'] = (
    most_endangered['EndangeredTotal'] / most_endangered['TotalRecords'] * 100
).round(1)

most_endangered = most_endangered.sort_values(
    ['CriticalCount', 'AvgUrgency'], ascending=False
)

print("\n🔴 6.1  MOST ENDANGERED SPECIES (sorted by Critical count & Urgency)")
print(most_endangered.to_string())



SECTION 6 — KEY INSIGHTS FROM WILDLIFE DISTRIBUTION ANALYSIS

🔴 6.1  MOST ENDANGERED SPECIES (sorted by Critical count & Urgency)
            CriticalCount  ExtinctCount  AvgPopulation  AvgUrgency  EndangeredTotal  TotalRecords  EndangeredRate%
Species                                                                                                           
Giraffe                 0             0       49347.44        0.04                0           544              0.0
Panda                   0             0       49902.79        0.04                0           549              0.0
Elephant                0             0       50010.46        0.03                0          1068              0.0
Kangaroo                0             0       49036.79        0.03                0           521              0.0
Leopard                 0             0       50722.49        0.03                0           560              0.0
Lion                    0             0       51439.63        0.

In [ ]:
# ── 6.2  Top 3 Most Endangered (highlighted) ─────────────────
top3 = most_endangered.head(3)
print("\n🚨 Top 3 Species Requiring Immediate Attention:")
for i, (sp, row) in enumerate(top3.iterrows(), 1):
    print(f"  {i}. {sp}")
    print(f"     Critical Observations : {int(row['CriticalCount'])}")
    print(f"     Extinct Observations  : {int(row['ExtinctCount'])}")
    print(f"     Avg Population        : {row['AvgPopulation']:,.0f}")
    print(f"     Avg Urgency Score     : {row['AvgUrgency']:.3f}")
    print(f"     Endangered Rate       : {row['EndangeredRate%']}%")


🚨 Top 3 Species Requiring Immediate Attention:
  1. Giraffe
     Critical Observations : 0
     Extinct Observations  : 0
     Avg Population        : 49,347
     Avg Urgency Score     : 0.040
     Endangered Rate       : 0.0%
  2. Panda
     Critical Observations : 0
     Extinct Observations  : 0
     Avg Population        : 49,903
     Avg Urgency Score     : 0.040
     Endangered Rate       : 0.0%
  3. Elephant
     Critical Observations : 0
     Extinct Observations  : 0
     Avg Population        : 50,010
     Avg Urgency Score     : 0.030
     Endangered Rate       : 0.0%


In [ ]:
# ── 6.3  Population Trends ───────────────────────────────────
print("\n" + "="*65)
print("6.3  POPULATION TRENDS")
print("="*65)

# Year-over-year change per species
pop_yoy = df.groupby(['Species', 'Year'])['PopulationEstimate'] \
             .mean().reset_index()
pop_yoy = pop_yoy.sort_values(['Species', 'Year'])
pop_yoy['YoY_Change%'] = pop_yoy.groupby('Species')['PopulationEstimate'] \
                                  .pct_change() * 100

avg_yoy = pop_yoy.groupby('Species')['YoY_Change%'].mean().round(2).sort_values()

print("\n📉 Species with GREATEST POPULATION DECLINE (avg YoY %):")
print(avg_yoy.head(5).to_string())

print("\n📈 Species with GREATEST POPULATION GROWTH (avg YoY %):")
print(avg_yoy.tail(5).sort_values(ascending=False).to_string())

# Overall population trend
first_year = df['Year'].min()
last_year  = df['Year'].max()
pop_first  = df[df['Year'] == first_year]['PopulationEstimate'].mean()
pop_last   = df[df['Year'] == last_year]['PopulationEstimate'].mean()
overall_change = ((pop_last - pop_first) / pop_first * 100).round(2)

print(f"\n📊 Overall Population Change ({first_year} → {last_year}): {overall_change:+.2f}%")
if overall_change < 0:
    print("   ⚠️  Overall declining trend across all species.")
else:
    print("   ✅  Overall recovering trend across all species.")


6.3  POPULATION TRENDS

📉 Species with GREATEST POPULATION DECLINE (avg YoY %):
Species
Panda         0.56
Tiger         0.79
Polar Bear    0.80
Elephant      0.89
Kangaroo      1.32

📈 Species with GREATEST POPULATION GROWTH (avg YoY %):
Species
Zebra      2.76
Leopard    2.61
Wolf       1.99
Giraffe    1.94
Lion       1.65

📊 Overall Population Change (1995 → 2025): -9.52%
   ⚠️  Overall declining trend across all species.


In [ ]:

# ── 6.4  Region-wise Distribution ────────────────────────────
print("\n" + "="*65)
print("6.4  REGION-WISE DISTRIBUTION")
print("="*65)

region_summary = df.groupby('Region').agg(
    TotalObservations  = ('Species', 'count'),
    UniqueSpecies      = ('Species', 'nunique'),
    AvgPopulation      = ('PopulationEstimate', 'mean'),
    TotalPopulation    = ('PopulationEstimate', 'sum'),
    EndangeredCount    = ('IsEndangered', 'sum'),
    CriticalCount      = ('ThreatLevel', lambda x: (x == 'Critical').sum()),
    ExtinctCount       = ('ThreatLevel', lambda x: (x == 'Extinct').sum()),
    AvgUrgency         = ('ConservationUrgency', 'mean')
).round(2)

region_summary['EndangeredRate%'] = (
    region_summary['EndangeredCount'] / region_summary['TotalObservations'] * 100
).round(1)

region_summary = region_summary.sort_values('AvgUrgency', ascending=False)

print("\n🌍 Full Region Summary (sorted by Avg Urgency):")
print(region_summary.to_string())

print("\n🏆 Highest Risk Region      :", region_summary['AvgUrgency'].idxmax())
print("🌱 Lowest Risk Region       :", region_summary['AvgUrgency'].idxmin())
print("🐾 Most Species Observed    :", region_summary['UniqueSpecies'].idxmax(),
      f"({int(region_summary['UniqueSpecies'].max())} species)")
print("📍 Most Total Observations  :", region_summary['TotalObservations'].idxmax(),
      f"({int(region_summary['TotalObservations'].max())} records)")





6.4  REGION-WISE DISTRIBUTION

🌍 Full Region Summary (sorted by Avg Urgency):
               TotalObservations  UniqueSpecies  AvgPopulation  TotalPopulation  EndangeredCount  CriticalCount  ExtinctCount  AvgUrgency  EndangeredRate%
Region                                                                                                                                                    
North America                864             10       49207.31         46648529                0              0             0        0.04              0.0
South America                924             10       48476.47         48621898                0              0             0        0.04              0.0
Asia                         955             10       51944.20         53866137                0              0             0        0.03              0.0
Africa                       975             10       49101.69         52833418                0              0             0        0.03         

In [ ]:
# ── 6.5  Habitat Risk Analysis ───────────────────────────────
print("\n" + "="*65)
print("6.5  HABITAT RISK ANALYSIS")
print("="*65)

habitat_risk = df.groupby('HabitatType').agg(
    TotalObservations = ('Species', 'count'),
    UniqueSpecies     = ('Species', 'nunique'),
    AvgPopulation     = ('PopulationEstimate', 'mean'),
    MinPopulation     = ('PopulationEstimate', 'min'),
    MaxPopulation     = ('PopulationEstimate', 'max'),
    EndangeredCount   = ('IsEndangered', 'sum'),
    CriticalCount     = ('ThreatLevel', lambda x: (x == 'Critical').sum()),
    ExtinctCount      = ('ThreatLevel', lambda x: (x == 'Extinct').sum()),
    AvgUrgency        = ('ConservationUrgency', 'mean')
).round(2)

habitat_risk['EndangeredRate%'] = (
    habitat_risk['EndangeredCount'] / habitat_risk['TotalObservations'] * 100
).round(1)

habitat_risk = habitat_risk.sort_values('AvgUrgency', ascending=False)

print("\n🏔️  Full Habitat Risk Summary (sorted by Avg Urgency):")
print(habitat_risk.to_string())

print("\n🔥 Most At-Risk Habitat     :", habitat_risk['AvgUrgency'].idxmax())
print("🌿 Safest Habitat           :", habitat_risk['AvgUrgency'].idxmin())
print("🌊 Highest Endangered Rate  :", habitat_risk['EndangeredRate%'].idxmax(),
      f"({habitat_risk['EndangeredRate%'].max()}%)")



6.5  HABITAT RISK ANALYSIS

🏔️  Full Habitat Risk Summary (sorted by Avg Urgency):
             TotalObservations  UniqueSpecies  AvgPopulation  MinPopulation  MaxPopulation  EndangeredCount  CriticalCount  ExtinctCount  AvgUrgency  EndangeredRate%
HabitatType                                                                                                                                                          
Arctic                     813             10       50184.40            204          99940                0              0             0        0.03              0.0
Desert                     749             10       49924.87             79          99945                0              0             0        0.03              0.0
Forest                    1592             10       49927.14             64          99979                0              0             0        0.03              0.0
Grassland                  807             10       50056.82            407          9

In [ ]:
# ── 6.6  Threat Level Summary ────────────────────────────────
print("\n" + "="*65)
print("6.6  THREAT LEVEL SUMMARY")
print("="*65)

threat_summary = df.groupby('ThreatLevel').agg(
    Count          = ('Species', 'count'),
    UniqueSpecies  = ('Species', 'nunique'),
    AvgPopulation  = ('PopulationEstimate', 'mean'),
    AvgUrgency     = ('ConservationUrgency', 'mean')
).round(2)

threat_summary['Share%'] = (
    threat_summary['Count'] / threat_summary['Count'].sum() * 100
).round(1)

threat_ord = ['Low', 'Moderate', 'High', 'Critical', 'Extinct', 'Unknown']
threat_summary = threat_summary.reindex(
    [t for t in threat_ord if t in threat_summary.index]
)

print(threat_summary.to_string())


6.6  THREAT LEVEL SUMMARY
             Count  UniqueSpecies  AvgPopulation  AvgUrgency  Share%
ThreatLevel                                                         
Unknown          0              0        50097.0        0.02     0.0


In [ ]:
# ── 6.7  Year-wise Highlights ────────────────────────────────
print("\n" + "="*65)
print("6.7  YEAR-WISE HIGHLIGHTS")
print("="*65)

yearly = df.groupby('Year').agg(
    AvgPopulation    = ('PopulationEstimate', 'mean'),
    EndangeredCount  = ('IsEndangered', 'sum'),
    TotalRecords     = ('Species', 'count'),
    UniqueSpecies    = ('Species', 'nunique'),
    AvgUrgency       = ('ConservationUrgency', 'mean')
).round(2)

yearly['EndangeredRate%'] = (
    yearly['EndangeredCount'] / yearly['TotalRecords'] * 100
).round(1)

print(yearly.to_string())

peak_endangered_yr = yearly['EndangeredRate%'].idxmax()
best_pop_yr        = yearly['AvgPopulation'].idxmax()
worst_pop_yr       = yearly['AvgPopulation'].idxmin()

print(f"\n📅 Year with Highest Endangered Rate : {peak_endangered_yr} "
      f"({yearly.loc[peak_endangered_yr, 'EndangeredRate%']}%)")
print(f"📅 Year with Highest Avg Population  : {best_pop_yr} "
      f"({yearly.loc[best_pop_yr, 'AvgPopulation']:,.0f})")
print(f"📅 Year with Lowest Avg Population   : {worst_pop_yr} "
      f"({yearly.loc[worst_pop_yr, 'AvgPopulation']:,.0f})")



6.7  YEAR-WISE HIGHLIGHTS
      AvgPopulation  EndangeredCount  TotalRecords  UniqueSpecies  AvgUrgency  EndangeredRate%
Year                                                                                          
1995       51800.00                0           219             10        0.03              0.0
1996       50245.80                0           207             10        0.03              0.0
1997       45625.33                0           192             10        0.04              0.0
1998       48383.59                0           199             10        0.04              0.0
1999       50247.60                0           197             10        0.04              0.0
2000       48150.08                0           190             10        0.03              0.0
2001       49002.29                0           210             10        0.03              0.0
2002       49500.82                0           211             10        0.03              0.0
2003       48858.63    

In [ ]:
# ── 6.8  Species–Habitat Hotspot Table ───────────────────────
print("\n" + "="*65)
print("6.8  SPECIES–HABITAT HOTSPOTS (High/Critical only)")
print("="*65)

hotspots = df[df['ThreatLevel'].isin(['High', 'Critical', 'Extinct'])].groupby(
    ['Species', 'HabitatType']
).agg(
    Count         = ('ThreatLevel', 'count'),
    AvgPopulation = ('PopulationEstimate', 'mean'),
    AvgUrgency    = ('ConservationUrgency', 'mean')
).round(2).sort_values('AvgUrgency', ascending=False)

print(hotspots.head(15).to_string())



6.8  SPECIES–HABITAT HOTSPOTS (High/Critical only)
Empty DataFrame
Columns: [Count, AvgPopulation, AvgUrgency]
Index: []


ANALYSIS COMPLETE — Wildlife Species Distribution Analysis